**What is an Embedding?**

An embedding is a numerical representation of text, images, audio, or other data in the form of a vector (a list of numbers).

Computers cannot directly understand the meaning of human language. Embeddings help computers compare the meaning of different texts by converting them into numerical vectors.

**Why Do We Need Embeddings?**

Embeddings are useful because they help us:
- Find text with similar meanings.
- Perform semantic search.
- Retrieve relevant documents in RAG.
- Build chatbots and question-answering systems.
- Recommend similar products or content.
- Compare text, images, and other supported data.

# **Types of Embeddings:-**

Embeddings can be broadly classified into two categories:

# **A. Sparse vs. Dense Embeddings**

## Sparse Embeddings (e.g., TF-IDF, BoW, BM25):

- High-dimensional vectors where most values are zero.

- Based on exact word counts and frequency.

- Limitation: They do not understand synonyms or semantic context (e.g., "bank" as a river bank vs. a financial bank).

## Dense Embeddings (e.g., Sentence-Transformers, OpenAI Embeddings):

- Low-dimensional vectors (e.g., 384, 768, or 1536 dimensions) where every number carries meaningful semantic weight.

- Captures deep contextual relationships.

# B. **Static vs. Dynamic (Contextual) Embeddings**
## Static Embeddings (e.g., Word2Vec, GloVe):

- Each unique word has a single fixed vector, regardless of context. (e.g., the word "apple" has the exact same vector whether referring to the fruit or the company).

- Dynamic / Contextual Embeddings (e.g., BERT, Sentence-Transformers, LLM embeddings):

- The vector changes depending on the surrounding context of the sentence or paragraph, resolving ambiguity (polysemy).

In [ ]:

!pip install -q sentence-transformers scikit-learn pandas


In [ ]:
# Import the SentenceTransformer class to convert text sentences into vector embeddings
from sentence_transformers import SentenceTransformer

# Define the name of the pre-trained model we want to download and use
model_name = "sentence-transformers/all-MiniLM-L6-v2"

# Load the specified model so it is ready to generate embeddings for our texts
model = SentenceTransformer(model_name)

# Print a success message to confirm that the model finished loading without errors
print("Model loaded successfully!")

# Print the dimension size of the vectors generated by this model (which is 384)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!
Embedding dimension: 384


/tmp/ipykernel_2055/3168969540.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [ ]:
# Convert text into Embeddings

In [ ]:
# Create a list of sample sentences that we want to convert into vector embeddings
sentences = [
    "I love machine learning",
    "I enjoy studying artificial intelligence",
    "Python is a programming language",
    "I want to learn deep learning",
    "The weather is very hot today"
]

# Use our loaded model to convert the list of text sentences into numerical vectors
embeddings = model.encode(sentences) #model.encode() converts each sentence into a vector.

# Print the total number of sentences we processed
print("Number of sentences:", len(sentences))

# Print the shape of our embeddings matrix to see the number of rows and columns (5, 384)
print("Embedding shape:", embeddings.shape)

# Print a label to indicate we are displaying the first sentence from our list
print("\nFirst sentence:")

# Display the actual text of the first sentence in our list
print(sentences[0])

# Print a label to show we are displaying the start of the first vector
print("\nFirst 10 embedding values:")

# Print only the first 10 numerical values of the 384-dimensional vector for the first sentence
print(embeddings[0][:10])

Number of sentences: 5
Embedding shape: (5, 384)

First sentence:
I love machine learning

First 10 embedding values:
[-0.04363634 -0.05905436  0.08201231 -0.01076716  0.06119589 -0.05515967
 -0.00290433 -0.03672321  0.02139721 -0.00756044]


In [ ]:
# Calculate semantic similarity

In [ ]:
# Import cosine_similarity to measure the directional similarity between different vectors
from sklearn.metrics.pairwise import cosine_similarity

# Import the pandas library to organize and display our data in a structured table
import pandas as pd

# Calculate the similarity score between every pair of sentence embeddings
similarity_matrix = cosine_similarity(embeddings)

# Convert the raw numerical matrix into a formatted table with sentence texts as row and column names
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=sentences,
    columns=sentences
)

# Display the similarity table with numbers rounded to 3 decimal places for better readability
display(similarity_df.round(3))

,I love machine learning,I enjoy studying artificial intelligence,Python is a programming language,I want to learn deep learning,The weather is very hot today
I love machine learning,1.000,0.567,0.236,0.479,0.024
I enjoy studying artificial intelligence,0.567,1.000,0.354,0.445,-0.047
Python is a programming language,0.236,0.354,1.000,0.191,0.067
I want to learn deep learning,0.479,0.445,0.191,1.000,-0.021
The weather is very hot today,0.024,-0.047,0.067,-0.021,1.000


In [ ]:
# Example-2

In [ ]:
# Create a list of reference documents that will act as our small knowledge base
documents = [
    "Machine learning enables computers to learn from data.",
    "Deep learning uses neural networks with multiple layers.",
    "Python is widely used for data science and automation.",
    "RAG retrieves relevant documents before generating an answer.",
    "Embeddings represent text as numerical vectors.",
    "A database stores and organizes application information.",
    "Computer vision helps machines understand images.",
    "Natural language processing works with human language."
]

# Convert all reference documents into normalized vectors (length = 1) for fast similarity matching
document_embeddings = model.encode(
    documents,
    normalize_embeddings=True #Normalization makes similarity calculations using the dot product convenient and equivalent to cosine similarity for nonzero vectors.
)

# Print the total number of documents we have successfully indexed
print("Documents indexed:", len(documents))

# Print the final shape of our document vector matrix (8 documents, 384 dimensions each)
print("Vector matrix shape:", document_embeddings.shape)

Documents indexed: 8
Vector matrix shape: (8, 384)


In [ ]:
# Import the numpy library to handle mathematical operations like sorting indices
import numpy as np

# Define a custom function to perform semantic search using a search query and a target result count
def semantic_search(query, top_k=3):
    # Convert the input search query text into a normalized vector embedding
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    # Calculate similarity scores by performing a matrix multiplication between document vectors and the query vector
    scores = (
        document_embeddings @ query_embedding[0]
    )

    # Sort the scores in descending order and select the indices of the top matches
    top_indices = np.argsort(scores)[::-1][:top_k]

    # Initialize an empty list to store the final matched results
    results = []

    # Loop through each of the top matched document indices
    for idx in top_indices:
        # Append the document text and its calculated similarity score to our results list
        results.append({
            "document": documents[idx],
            "similarity_score": round(float(scores[idx]), 4)
        })

    # Convert the accumulated results list into a clean pandas DataFrame and return it
    return pd.DataFrame(results)


# Call our search function to find the top 3 best matching documents for a sample question
results = semantic_search(
    "How can AI understand pictures?",
    top_k=3
)

# Display the formatted search results on the screen
display(results)

,document,similarity_score
0,Computer vision helps machines understand images.,0.6862
1,Machine learning enables computers to learn fr...,0.4369
2,Deep learning uses neural networks with multip...,0.4221


In [ ]:
semantic_search("How does RAG retrieve information?", top_k=3)

,document,similarity_score
0,RAG retrieves relevant documents before genera...,0.6652
1,A database stores and organizes application in...,0.2561
2,Natural language processing works with human l...,0.2335


In [ ]:
semantic_search("I want to learn neural networks", top_k=3)

,document,similarity_score
0,Deep learning uses neural networks with multip...,0.5131
1,Machine learning enables computers to learn fr...,0.3734
2,Computer vision helps machines understand images.,0.3321


In [ ]:
semantic_search("Context length is the maximum input a model can process, while embedding dimension is the number of values in the output vector. These are two different things. Retrieval quality also depends on your dataset, chunking strategy, and evaluation method.",top_k=4)

,document,similarity_score
0,Embeddings represent text as numerical vectors.,0.4753
1,Deep learning uses neural networks with multip...,0.2433
2,A database stores and organizes application in...,0.1687
3,RAG retrieves relevant documents before genera...,0.1466


In [ ]:
#example 3

In [ ]:
with open("novel.txt", "r", encoding="utf-8") as f:
    content = f.read()

documents = [p.strip() for p in content.split("\n\n") if p.strip()]

document_embeddings = model.encode(
    documents,
    normalize_embeddings=True
)

print("Documents indexed:", len(documents))
print("Vector matrix shape:", document_embeddings.shape)

Documents indexed: 818
Vector matrix shape: (818, 384)


In [ ]:
import numpy as np
import pandas as pd

def semantic_search(query, top_k=3):
    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    )

    scores = document_embeddings @ query_embedding

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append({
            "document": documents[idx],
            "similarity_score": round(float(scores[idx]), 4)
        })

    return pd.DataFrame(results)

In [ ]:
semantic_search("For, with all her knowledge of history, Alice had no very clear notion how long ago anything had happened.",top_k=3)

,document,similarity_score
0,Alice was silent.,0.6131
1,Alice said nothing: she had never been so much...,0.6130
2,“It proves nothing of the sort!” said Alice. “...,0.5952


In [ ]:
semantic_search("so that it was not easy to know when the race was over.",top_k=5)

,document,similarity_score
0,"First it marked out a race-course, in a sort o...",0.5665
1,The other guests had taken advantage of the Qu...,0.3588
2,"“The reason is,” said the Gryphon, “that they ...",0.3564
3,“I didn’t!” the March Hare interrupted in a gr...,0.3534
4,“What _is_ a Caucus-race?” said Alice; not tha...,0.3529
